In [1]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

import tensorflow as tf
import xgboost as xgb

2025-12-17 02:47:51.794487: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-17 02:47:51.794708: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-17 02:47:51.825801: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-17 02:47:52.454741: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off,

In [2]:
seed = 10
np.random.seed(seed)
tf.random.set_seed(seed)

data = pd.read_csv("heart_final.csv")

y = data["HeartDisease"].to_numpy()
X = data.drop(["HeartDisease"], axis=1)

num_cols = ["Age", "RestingBP", "Cholesterol", "MaxHR", "Oldpeak"]

ct = ColumnTransformer(
    transformers=[("stand_scal", StandardScaler(), num_cols)],
    remainder="passthrough"
)

X = ct.fit_transform(X)

In [3]:
train_size = 0.7
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=train_size, random_state=2
)

X_train_tf = tf.convert_to_tensor(X_train, dtype=tf.float64)
X_test_tf  = tf.convert_to_tensor(X_test, dtype=tf.float64)
y_train_tf = tf.convert_to_tensor(y_train, dtype=tf.float64)
y_test_tf  = tf.convert_to_tensor(y_test, dtype=tf.float64)

Xtr = X_train_tf.numpy()
Xte = X_test_tf.numpy()
ytr = y_train_tf.numpy().astype(int)
yte = y_test_tf.numpy().astype(int)

2025-12-17 02:47:52.650247: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [4]:
Xtr = X_train_tf.numpy()
Xte = X_test_tf.numpy()
ytr = y_train_tf.numpy().astype(int)
yte = y_test_tf.numpy().astype(int)

dtrain = xgb.DMatrix(Xtr, label=ytr)
dtest  = xgb.DMatrix(Xte, label=yte)

params = {
    "objective": "binary:logistic",
    "eval_metric": ["logloss", "auc"],
    "max_depth": 4,
    "eta": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 1.0,
    "lambda": 1.0,     # L2 regularization
    "alpha": 0.0,      # L1 regularization
    "seed": seed,
}

In [5]:
evals = [(dtrain, "train"), (dtest, "test")]

bst = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=2000,
    evals=evals,
    early_stopping_rounds=50,
    verbose_eval=50
)

[0]	train-logloss:0.65425	train-auc:0.92994	test-logloss:0.67149	test-auc:0.89695
[50]	train-logloss:0.24065	train-auc:0.97991	test-logloss:0.34201	test-auc:0.93199
[100]	train-logloss:0.17026	train-auc:0.98968	test-logloss:0.32150	test-auc:0.93396
[124]	train-logloss:0.15205	train-auc:0.99269	test-logloss:0.32486	test-auc:0.93233


In [6]:
y_prob = bst.predict(dtest)                  # probabilities for class 1
threshold = 0.5
y_pred = (y_prob >= threshold).astype(int)   # hard labels

print("\n===== Results =====")
print("Accuracy:", accuracy_score(yte, y_pred))
print("ROC-AUC :", roc_auc_score(yte, y_prob))
print("\nConfusion matrix:\n", confusion_matrix(yte, y_pred))
print("\nClassification report:\n", classification_report(yte, y_pred, digits=4))

# --------- (Optional) Save model ---------
# bst.save_model("xgb_heart.json")



===== Results =====
Accuracy: 0.8514492753623188
ROC-AUC : 0.9323308270676691

Confusion matrix:
 [[107  26]
 [ 15 128]]

Classification report:
               precision    recall  f1-score   support

           0     0.8770    0.8045    0.8392       133
           1     0.8312    0.8951    0.8620       143

    accuracy                         0.8514       276
   macro avg     0.8541    0.8498    0.8506       276
weighted avg     0.8533    0.8514    0.8510       276



In [ ]:
y_prob = bst.predict(dtest)                  # probabilities for class 1
threshold = 0.5
y_pred = (y_prob >= threshold).astype(int)   # hard labels

print("\n===== Results =====")
print("Accuracy:", accuracy_score(yte, y_pred))
print("ROC-AUC :", roc_auc_score(yte, y_prob))
print("\nConfusion matrix:\n", confusion_matrix(yte, y_pred))
print("\nClassification report:\n", classification_report(yte, y_pred, digits=4))

# --------- (Optional) Save model ---------
# bst.save_model("xgb_heart.json")



===== Results =====
Accuracy: 0.8514492753623188
ROC-AUC : 0.9323308270676691

Confusion matrix:
 [[107  26]
 [ 15 128]]

Classification report:
               precision    recall  f1-score   support

           0     0.8770    0.8045    0.8392       133
           1     0.8312    0.8951    0.8620       143

    accuracy                         0.8514       276
   macro avg     0.8541    0.8498    0.8506       276
weighted avg     0.8533    0.8514    0.8510       276

